In [1]:
import warnings
warnings.simplefilter('ignore')

In [2]:
import os, sys
from typing import Optional

import polars as pl
import pandas as pd

REPO_DATASET_PATH = "/kaggle/input/olympiadlevelmaths4llm"
sys.path.append(REPO_DATASET_PATH + "/src")

# --- Runtime / platform guards ---
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["TIKTOKEN_ENCODINGS_BASE"] = (
    "/kaggle/usr/lib/aimo3_packages_offline/tiktoken_encodings"
)

# =============================================================================
# V2 CONFIG — only env vars that AIMO3Config.from_env() actually reads
# =============================================================================

# --- Prompts (override defaults from config.py) ---
os.environ["AIMO3_SYSTEM_PROMPT"] = (
    "You are an elite olympiad problem solver whose job is to find the correct integer answer as fast as possible. "
    "Optimize for efficient problem-solving, not long exposition. The final answer must be a verified integer in \\boxed{n}, where n is in [0, 99999].\n\n"
    "WORKFLOW:\n"
    "1. CLASSIFY FIRST: Identify the problem type quickly (number theory, combinatorics, algebra, geometry, recurrence, functional equation, counting, optimization). "
    "Choose the cheapest exact method before doing anything else.\n"
    "2. PICK THE BEST ALGORITHM: Prefer, in this order when applicable: direct formula/invariant > symmetry or monotonicity > recurrence/dynamic counting > algebraic reduction > modular argument > bounded search/brute force. "
    "Avoid human-style full proofs unless they are needed to pin down the integer.\n"
    "3. USE PYTHON EARLY: Compute small cases, test conjectures, factor integers, search for patterns, and reject bad approaches quickly. "
    "If a brute-force method scales poorly, use it only on small instances to discover structure, then switch to a faster exact method.\n"
    "4. THINK ABOUT COMPLEXITY: Before heavy computation, estimate whether the approach is feasible. Reject approaches that obviously blow up. "
    "Prefer algorithms that are exact and simple to verify.\n"
    "5. VERIFY INDEPENDENTLY: Once you have a candidate answer, confirm it by a second method: substitution, invariant check, alternative derivation, modular check, or brute-force validation on small cases.\n"
    "6. BE CONCISE: Do not repeat the problem statement, do not write long essays, and do not explore many equivalent methods. "
    "Use one promising line of attack, switch only if it fails, and stop once the answer is independently verified.\n"
    "7. OUTPUT RULE: Return only the final verified integer in \\boxed{n}. Never guess. If unsure, compute more small cases or verify again."
)

os.environ["AIMO3_TOOL_PROMPT"] = (
    "Execute Python code in a stateful Jupyter notebook. Use the tool to discover the best algorithm, not just to do arithmetic.\n\n"
    "Pre-loaded: math, numpy (np), sympy (sp), mpmath (mp), itertools, collections, fractions.Fraction, random, ortools (if available).\n\n"
    "Tool strategy:\n"
    "1. Start with the smallest exact experiment that can eliminate or confirm an approach.\n"
    "2. Prefer short scripts that print only decisive quantities: candidate counts, residues, recurrences, factorizations, counterexamples, or the final checked value.\n"
    "3. For parameterized problems, compute several small cases first and print them clearly. Use those data to infer a formula, invariant, or recurrence.\n"
    "4. Before large loops, estimate complexity. Avoid naive enumeration when the state space is too large.\n"
    "5. Use sympy for symbolic manipulation, solving, factoring, recurrences, and number theory helpers. Use modular arithmetic for large integers.\n"
    "6. When checking a final candidate, print a concise verification summary and print VERIFY_OK only if the computation supports the candidate.\n"
    "7. Always use print() to show results.\n"
    "8. Default timeout is 30s. For heavy computation add '# timeout: 120' as the FIRST line.\n"
    "9. Final answer must be an integer in [0, 99999]."
)

os.environ["AIMO3_PREFERENCE_PROMPT"] = (
    "Solve for the final integer efficiently. Prefer exact structure over long prose. "
    "Use Python to test small cases, find patterns, estimate complexity, and verify the final candidate independently. "
    "If brute force is only feasible for small n, use it to infer the right algorithm, then switch to the scalable exact method. "
    "Keep reasoning compact and ensure the final answer is in [0, 99999]."
)

# --- Model / server ---
os.environ["AIMO3_MODEL_PATH"] = "/kaggle/input/gpt-oss-120b/transformers/default/1"
os.environ["AIMO3_SERVED_MODEL_NAME"] = "gpt-oss"
os.environ["AIMO3_REUSE_EXISTING_SERVER"] = "1"
os.environ["AIMO3_SERVER_TIMEOUT"] = "1500"
os.environ["AIMO3_REQUIRE_CUDA"] = "1"

# Cold-start speedup
os.environ["AIMO3_PRELOAD_MODEL_WEIGHTS"] = "1"
os.environ["AIMO3_PRELOAD_MODEL_WORKERS"] = "8"

# --- Display / tracing ---
os.environ["AIMO3_DISPLAY_CANDIDATES"] = "1"
os.environ["AIMO3_TRACE"] = "1"
os.environ["AIMO3_TRACE_ATTEMPTS"] = "0"
os.environ["AIMO3_TRACE_ENV"] = "1"
os.environ["AIMO3_TRACE_ENV_PACKAGES"] = "sympy,numpy,mpmath,jupyter_client,ortools,z3-solver"
os.environ["AIMO3_TRACE_INCLUDE_PROBLEM_TEXT"] = "0"

# --- Core decoding / capacity ---
os.environ["AIMO3_SEED"] = "42"
os.environ["AIMO3_SEARCH_TOKENS"] = "32"
os.environ["AIMO3_CONTEXT_TOKENS"] = "65536"
os.environ["AIMO3_BATCH_SIZE"] = "128"
os.environ["AIMO3_GPU_MEMORY_UTILIZATION"] = "0.96"

# --- Sandbox/tooling ---
os.environ["AIMO3_JUPYTER_TIMEOUT"] = "25"
os.environ["AIMO3_SANDBOX_TIMEOUT"] = "10"

# --- Time budgeting ---
os.environ["AIMO3_PROBLEMS_TOTAL"] = "50"
os.environ["AIMO3_NOTEBOOK_LIMIT"] = "17700"
os.environ["AIMO3_BASE_PROBLEM_TIMEOUT"] = "280"
os.environ["AIMO3_HIGH_PROBLEM_TIMEOUT"] = "1500"

# --- Attempt scheduling ---
os.environ["AIMO3_ATTEMPTS"] = "5"
os.environ["AIMO3_WORKERS"] = "8"
os.environ["AIMO3_TURNS"] = "128"
os.environ["AIMO3_EARLY_STOP"] = "4"
os.environ["AIMO3_EARLY_STOP_MIN_VERIFIED"] = "0"

# --- Extraction ---
os.environ["AIMO3_STRICT_FALLBACK_EXTRACTION"] = "1"

# --- Decoding knobs ---
os.environ["AIMO3_TEMPERATURE"] = "1.0"
os.environ["AIMO3_MIN_P"] = "0.02"
os.environ["AIMO3_TOP_P"] = "0.98"
os.environ["AIMO3_TOP_K"] = "-1"

os.environ["AIMO3_VERIFY_PHASE_ENABLED"] = "0"
os.environ["AIMO3_VERIFY_DISABLE_GLOBALLY_IF_ALL_UNKNOWN"] = "1"
os.environ["AIMO3_VERIFY_ATTEMPTS_PER_CANDIDATE"] = "2"
os.environ["AIMO3_VERIFY_TOP_K_CANDIDATES"] = "2"
os.environ["AIMO3_VERIFY_TIMEOUT"] = "30"
os.environ["AIMO3_VERIFY_TEMPERATURE"] = "0.3"
os.environ["AIMO3_PYTHON_TOOL_VERIFY_REQUIRE_MARKER"] = "0"

# --- Time Management Approach ---
# - 'equal': use equal share of remaining time
# - 'base': use configured base_timeout_s for every problem
# - 'avg': use rolling average
# - 'cumulative': add carryover from previous problems
# - 'hybrid': take max(equal, avg, base) (default behavior)
os.environ["AIMO3_BUDGET_STRATEGY"] = "cumulative"
os.environ["AIMO3_BASE_TIMEOUT_S"] = ""  # unset to avoid overriding base_problem_timeout
os.environ["AIMO3_CARRYOVER_ENABLED"] = "1"
os.environ["AIMO3_CUMULATIVE_DISTRIBUTE"] = "0"

os.environ["AIMO3_WICKELGREN"] = "1"
os.environ["AIMO3_TRACE_ATTEMPTS"] = "1"
os.environ["AIMO3_TRACE_ATTEMPTS_MAX_CHARS"] = "60000"

os.environ["AIMO3_FILTER_TO_VERIFIED_IF_ANY"] = "0"
os.environ["AIMO3_ENTROPY_WEIGHTING"] = "0"
os.environ["AIMO3_RANKING_STRATEGY"] = "votes_then_verified"

# --- CPU retriever (v2, v1-compatible env names) ---
# Set this path to your mounted KB dir containing concepts.json or concepts.pkl
os.environ["AIMO3_RETRIEVER_ENABLED"] = "0"
os.environ["AIMO3_RETRIEVER_KB_PATH"] = "/kaggle/input/olympiadlevelmaths4llmdb/OlympiadLevelMaths4LLMDB/knowledge_base"
os.environ["AIMO3_RETRIEVER_CPU_ONLY"] = "1"
os.environ["AIMO3_RETRIEVER_TOP_K"] = "5"
os.environ["AIMO3_RETRIEVER_MIN_SCORE"] = "0.08"
os.environ["AIMO3_RETRIEVER_INCLUDE_EXAMPLES"] = "1"
os.environ["AIMO3_RETRIEVER_INCLUDE_DEFINITIONS"] = "1"
os.environ["AIMO3_RETRIEVER_WARMUP_ON_INIT"] = "1"
os.environ["AIMO3_RETRIEVER_MODEL_PATH"] = "/kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2"

os.environ["AIMO3_ADAPTIVE_BUDGET_FLEX_POOL_FRACTION"] = "0"

# --- Meta Learning ---
os.environ["AIMO3_META_LEARNING_ENABLED"] = "0"
os.environ["AIMO3_META_LEARNING_SIMILARITY_THRESHOLD"] = "0.3"

os.environ["AIMO3_Z3_TOOL_ENABLED"] = "0"

# Misc
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

MODEL_PATH = os.getenv("AIMO3_MODEL_PATH", "")
print(
    f"Config: BASE_TIMEOUT={os.environ.get('AIMO3_BASE_PROBLEM_TIMEOUT')}s, HIGH_TIMEOUT={os.environ.get('AIMO3_HIGH_PROBLEM_TIMEOUT')}s"
)
print(
    f"Config: EARLY_STOP={os.environ.get('AIMO3_EARLY_STOP')}, MIN_VERIFIED={os.environ.get('AIMO3_EARLY_STOP_MIN_VERIFIED')}"
)
print(
    f"Config: ATTEMPTS={os.environ.get('AIMO3_ATTEMPTS')}, WORKERS={os.environ.get('AIMO3_WORKERS')}, TURNS={os.environ.get('AIMO3_TURNS')}"
)
print(
    f"Config: RANKING_STRATEGY={os.environ.get('AIMO3_RANKING_STRATEGY')}, FILTER_TO_VERIFIED_IF_ANY={os.environ.get('AIMO3_FILTER_TO_VERIFIED_IF_ANY')}"
)
print(
    f"Config: RETRIEVER_ENABLED={os.environ.get('AIMO3_RETRIEVER_ENABLED')}, RETRIEVER_KB_PATH={os.environ.get('AIMO3_RETRIEVER_KB_PATH')}"
)
MODEL_PATH

Config: BASE_TIMEOUT=280s, HIGH_TIMEOUT=1500s
Config: EARLY_STOP=4, MIN_VERIFIED=0
Config: ATTEMPTS=5, WORKERS=8, TURNS=128
Config: RANKING_STRATEGY=votes_then_verified, FILTER_TO_VERIFIED_IF_ANY=0
Config: RETRIEVER_ENABLED=0, RETRIEVER_KB_PATH=/kaggle/input/olympiadlevelmaths4llmdb/OlympiadLevelMaths4LLMDB/knowledge_base


'/kaggle/input/gpt-oss-120b/transformers/default/1'

In [3]:
# --- First-wave fast-answer mode (helps capture easy consensus before full reasoning) ---
os.environ["AIMO3_ANSWER_ONLY_ATTEMPTS"] = "4"
os.environ["AIMO3_ANSWER_ONLY_PROMPT"] = (
    "You are an IMO-level mathematician solving for a single integer answer. Think silently and optimize for speed and correctness. "
    "Internally do quick triage: classify the problem, choose the cheapest exact method, test small cases mentally or computationally if needed, and avoid long proof-style reasoning. "
    "Prefer invariant/formula/symmetry arguments over brute force; if brute force is only useful on tiny cases, use it only to infer structure. "
    "Output only the final verified integer answer in \\boxed{number}. If uncertain, keep reasoning internally until one boxed integer is justified."
)

# Re-enable entropy-weighted tie-breaking to recover a confidence signal across attempts.
os.environ["AIMO3_ENTROPY_WEIGHTING"] = "1"

print(
    f"Config: ANSWER_ONLY_ATTEMPTS={os.environ.get('AIMO3_ANSWER_ONLY_ATTEMPTS')}, "
    f"ENTROPY_WEIGHTING={os.environ.get('AIMO3_ENTROPY_WEIGHTING')}"
)

Config: ANSWER_ONLY_ATTEMPTS=4, ENTROPY_WEIGHTING=1


In [4]:
from olympiad_llm.aimo3.v2.cleanup import environ_setup_parallel, wait_all

# Start environment setup (uninstall + offline install + model warmup) ALL IN PARALLEL.
# This overlaps pip install with model cache warmup, saving ~30-60s on cold start.
ENVIRON_SETUP = environ_setup_parallel(warm_model=True, model_workers=8)
print("Started parallel environment setup:")
print("  - pip uninstall conflicts (background)")
print("  - pip install required packages (background)")
print("  - Model weight cache warmup (background)")
print("Will wait for completion lazily when the solver is first needed.")

Started parallel environment setup:
  - pip uninstall conflicts (background)
  - pip install required packages (background)
  - Model weight cache warmup (background)
Will wait for completion lazily when the solver is first needed.


In [5]:
import threading
from olympiad_llm.aimo3.v2.config import AIMO3Config
from olympiad_llm.aimo3.v2.runner import build_solver, run_kaggle_inference
from olympiad_llm.aimo3.v2.cleanup import wait_all

# IMPORTANT for Kaggle: don't block notebook execution on vLLM cold-start here.
# We'll build the solver lazily on the first predict() call.
cfg = AIMO3Config.from_env()
solver = None
_solver_lock = threading.Lock()

def get_solver():
    global solver
    if solver is not None:
        return solver
    with _solver_lock:
        if solver is not None:
            return solver
        # Wait for ALL parallel setup tasks to finish (pip + model warmup).
        if "ENVIRON_SETUP" in globals() and isinstance(ENVIRON_SETUP, dict):
            wait_all(ENVIRON_SETUP, timeout=180)
            print("✓ Parallel setup complete (pip + model cache warmup)")
        solver = build_solver(cfg)
        return solver

print("Lazy solver configured. Inference server can start now.")

Lazy solver configured. Inference server can start now.


In [6]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    global correct_count, total_count, predictions, ground_truth
    
    question_id = id_.item(0)
    question_text = question.item(0)
    
    print("------")
    print(f"ID: {question_id}")
    
    # Build solver only when needed (keeps Kaggle inference server startup fast).
    s = get_solver()
    final_answer = s.solve_problem(question_text)
    predictions[question_id] = final_answer

    # Check accuracy if ground truth available (local runs only).
    total_count += 1
    if question_id in ground_truth:
        gt = ground_truth[question_id]
        is_correct = (final_answer == gt)
        if is_correct:
            correct_count += 1
        status = "✅" if is_correct else "❌"
        print(f"Answer: {final_answer} | Ground Truth: {gt} | {status}")
        print(f"📊 Running Accuracy: {correct_count}/{total_count} ({100*correct_count/total_count:.1f}%)")
    else:
        print(f"Answer: {final_answer}")
    
    print("------\n")
    
    return pl.DataFrame({'id': question_id, 'answer': final_answer})

In [7]:
from olympiad_llm.aimo3.prepare import prepare_reference_csv

In [8]:
# Load ground truth only for local testing (avoid delaying server start in competition reruns).
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    ground_truth = {}
else:
    ground_truth, _ = prepare_reference_csv(
        "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv",
        # "/kaggle/input/olympiadlevelmaths4llmdb/inmo_1986.csv",
        # problem_ids=["dd7f5e", "86e8e5"],
        # problem_ids=["86e8e5"],
    )

# Track predictions for accuracy calculation
predictions = {}
correct_count = 0
total_count = 0

In [9]:
inference_server = run_kaggle_inference(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(("reference.csv",))

------
ID: 86e8e5
✓ Parallel setup complete (pip + model cache warmup)

Problem: Let $n \geq 6$ be a positive integer. We call a positive integer $n$-Norwegian if it has three distinct positive divisors whose sum is equal to $n$. Let $f(n)$ denote the smallest $n$-Norwegian positive integer. Let $M=3^{2025!}$ and for a non-negative integer $c$ define 
\begin{equation*}
    g(c)=\frac{1}{2025!}\left\lfloor \frac{2025! f(M+c)}{M}\right\rfloor.
\end{equation*}
We can write 
\begin{equation*}
    g(0)+g(4M)+g(1848374)+g(10162574)+g(265710644)+g(44636594)=\frac{p}{q}
\end{equation*}
where $p$ and $q$ are coprime positive integers. What is the remainder when $p+q$ is divided by $99991$?

Budget: 354.00s | [Budget] 0/50 done | Remaining: 17700s | Flex: 0s/0s | Avg: 280s | Next: 354s | Extensions: 0



,Attempt,Answer,ToolVerified,PyCalls,Timeouts,PyErrors,Tokens,Entropy,Snippet,LastError
0,1,None,False,23,0,1,39620,0.730775,"...nce N huge, it's very likely that among sma...",----------------------------------------------...
1,2,None,False,48,1,1,34675,0.700403,...sult = (num + den) % mod\nresult\nanalysisT...,[ERROR] Execution timed out after 25s. TIP: Fo...
2,3,None,False,28,0,1,38556,0.711127,...s total sum = (4/5 + 10/11 + 28/29 + 16/17)...,----------------------------------------------...
3,4,None,False,18,0,1,39573,0.770757,"...for c=10, f=24. M=27, c=10 => n=37, f=24 = ...",----------------------------------------------...
4,5,None,False,0,0,0,0,NaN,,None



Final Answer: 0 (no valid candidates)

Answer: 0 | Ground Truth: 8687 | ❌
📊 Running Accuracy: 0/1 (0.0%)
------

------
ID: 9c1c5f

Problem: Let $f \colon \mathbb{Z}_{\geq 1} \to \mathbb{Z}_{\geq 1}$ be a function such that for all positive integers $m$ and $n$, 
\begin{equation*}
    f(m) + f(n) = f(m + n + mn).
\end{equation*}
Across all functions $f$ such that $f(n) \leq 1000$ for all $n \leq 1000$, how many different values can $f(2024)$ take?

Budget: 353.98s | [Budget] 1/50 done | Remaining: 17345s | Flex: 0s/0s | Avg: 355s | Next: 354s | Extensions: 0



,Attempt,Answer,ToolVerified,PyCalls,Timeouts,PyErrors,Tokens,Entropy,Snippet,LastError
0,1,580,True,13,0,0,9476,0.802270,... numbers between 6 and 1164 inclusive. So t...,None
1,2,580,True,6,0,0,10750,0.801646,...for f(2024) across all such functions. Sinc...,None
2,3,580,False,17,0,2,12589,0.785986,"...st_pair(a,b, trials=1000):\n primes=buil...",----------------------------------------------...
3,4,580,True,22,0,0,12326,0.807760,"...function f. For any feasible (a,b) we can p...",None



Final Answer: 580 (votes=4, verified=3)

Answer: 580 | Ground Truth: 580 | ✅
📊 Running Accuracy: 1/2 (50.0%)
------

------
ID: 42d360

Problem: On a blackboard, Ken starts off by writing a positive integer $n$ and then applies the following move until he first reaches $1$. Given that the number on the board is $m$, he chooses a base $b$, where $2 \leq b \leq m$, and considers the unique base-$b$ representation of $m$,
\begin{equation*}
    m = \sum_{k = 0}^\infty a_k \cdot b^k
\end{equation*}
where $a_k$ are non-negative integers and $0 \leq a_k < b$ for each $k$. Ken then erases $m$ on the blackboard and replaces it with $\sum\limits_{k = 0}^\infty a_k$.

Across all choices of $1 \leq n \leq 10^{10^5}$, the largest possible number of moves Ken could make is $M$. What is the remainder when $M$ is divided by $10^{5}$?

Budget: 605.37s | [Budget] 2/50 done | Remaining: 17242s | Flex: 0s/0s | Avg: 229s | Next: 359s | Extensions: 0



...


Final Answer: 160 (votes=3, verified=0)

Answer: 160 | Ground Truth: 160 | ✅
📊 Running Accuracy: 7/8 (87.5%)
------

------
ID: 424e18

Problem: A tournament is held with $2^{20}$ runners each of which has a different running speed. In each race, two runners compete against each other with the faster runner always winning the race. The competition consists of $20$ rounds with each runner starting with a score of $0$. In each round, the runners are paired in such a way that in each pair, both runners have the same score at the beginning of the round. The winner of each race in the $i^{\text{th}}$ round receives $2^{20-i}$ points and the loser gets no points.

At the end of the tournament, we rank the competitors according to their scores. Let $N$ denote the number of possible orderings of the competitors at the end of the tournament. Let $k$ be the largest positive integer such that $10^k$ divides $N$. What is the remainder when $k$ is divided by $10^{5}$?

Budget: 1250.54s | [Budget] 

,Attempt,Answer,ToolVerified,PyCalls,Timeouts,PyErrors,Tokens,Entropy,Snippet,LastError
0,1,62140,False,6,1,1,13527,0.905494,"...140. So k = 262,140.\n\nNow we need remaind...",[ERROR] Execution timed out after 25s. TIP: Fo...
1,2,21818,True,10,0,0,22481,0.823737,"...r m=20, our computed valuations v2=524287, ...",None
2,3,21818,True,16,0,0,21313,0.749896,...ndent across nodes? Yes.\n\nThus final answ...,None
3,4,21818,False,21,0,4,31274,0.763125,...e lexicographic order.\n\nThus the mapping ...,----------------------------------------------...
4,5,21818,False,21,0,1,21829,0.803673,...ow produce final answer.\n\nOne more check:...,----------------------------------------------...



Final Answer: 21818 (votes=4, verified=2)

Answer: 21818 | Ground Truth: 21818 | ✅
📊 Running Accuracy: 8/9 (88.9%)
------

------
ID: a295e9

Problem: A $500 \times 500$ square is divided into $k$ rectangles, each having integer side lengths. Given that no two of these rectangles have the same perimeter, the largest possible value of $k$ is $\mathcal{K}$. What is the remainder when $k$ is divided by $10^{5}$?

Budget: 1253.90s | [Budget] 9/50 done | Remaining: 15413s | Flex: 0s/0s | Avg: 254s | Next: 376s | Extensions: 0



,Attempt,Answer,ToolVerified,PyCalls,Timeouts,PyErrors,Tokens,Entropy,Snippet,LastError
0,1,520,True,18,0,0,24565,0.889615,...on for N=500 to verify existence (maybe via...,None
1,2,520,True,13,0,0,59294,0.836468,...rhaps rows can have width less than 500 if ...,None
2,3,520,True,16,0,0,35994,0.848259,"...rea cannot be less than minimal, so we cann...",None
3,4,520,False,0,0,0,19796,0.932938,"..., we need total area exactly 250k, which is...",None



Final Answer: 520 (votes=4, verified=3)

Answer: 520 | Ground Truth: 520 | ✅
📊 Running Accuracy: 9/10 (90.0%)
------

